<img src="./ccsf.png" alt="CCSF Logo" width=200px style="margin:0px -5px">

# Lecture 36: Evaluating Classifiers

Associated Textbook Sections: [17.5](https://ccsf-math-108.github.io/textbook/chapters/17/5/Accuracy_of_the_Classifier.html)

---

## Outline

* [Implementing a Classifier](#Implementing-a-Classifier)
* [k-Nearest Neighbors Classifier](#k-Nearest-Neighbors-Classifier)
* [Evaluation](#Evaluation)
* [Picking `k`](#Picking-k)
* [Example: Chronic Kidney Disease (CKD)](#Example:-Chronic-Kidney-Disease-(CKD))
* [Standardizing Units](#Standardizing-Units)


---

## Set Up the Notebook

In [ ]:
from datascience import *
import numpy as np
%matplotlib inline
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')

---

## Implementing a Classifier

---

### The Process

In the previous lecture we learned about how to create a classifier from a training set, and to use a test set to make predictions. In this lecture we will learn how to evaluate, and improve, our classifier. One technique to improve our classifier will require us to add a "validation set" branch to the flowchart below.

```mermaid
graph TD
    A["Population"] --> B["Sample with labels"]

    B -->|"x% of Sample"| C["Create Training Set"]
    B -->|"y% of Sample"| V["Create Validation Set"]
    B -->|"100-x-y% of Sample"| D["Create Test Set"]

    C --> E["Train classifier on Training Set"]
    E --> F["Fine-Tune Classifier using Validation Set"]
    V --> F

   F --> H["Evaluate Classifier with Test Set"]
   D --> H
   
    H --> I["Apply Model"]
```

---

### Start with a Representative Sample

<img src="./dog_wolf.png" width=60%>

* The training, validation, and test sets must accurately represent the population on which you use your classifier
* Overfitting happens when a classifier does very well on the training set, but can't do as well on the test set

---

### Accuracy of a Classifier

* The accuracy of a classifier on a labeled data set is the proportion of examples in the test set that are labeled correctly
* Need to compare classifier predictions to true labels
* If the labeled data set is sampled at random from a population, then we can infer accuracy on that population
* While the accuracy of the classifier is the goal, we use the validation set to "fine-tune" our classifier, by determining the optimal `k`
* We must avoid ever using the test or validation sets for training, and we must avoid ever using the test set for determining the optimal `k` to avoid data leakage
* Data leakage is when data is trained on information that would not be available in a real-world prediction scenario

---

## k-Nearest Neighbors Classifier

---

### Rebuild the Classifier

Here are the functions from the last lecture:

In [ ]:
def distance(pt1, pt2):
    """Return the distance between two points, represented as arrays"""
    return np.sqrt(sum((pt1 - pt2)**2))
    
def row_distance(row1, row2):
    """Return the distance between two numerical rows of a table"""
    return distance(np.array(row1), np.array(row2))
    
def distances(training, example):
    """
    Compute distance between example row and every row in training.
    Return training augmented with Distance column
    """
    distances = make_array()
    features_only = training.drop('Class')
    
    for row in features_only.rows:
        distances = np.append(distances, row_distance(row, example))
        
    return training.with_column('Distance_to_ex', distances)
    
def closest(training, example, k):
    """
    Return a table of the k closest neighbors to example
    """
    return distances(training, example).sort('Distance_to_ex').take(np.arange(k))
    
def majority_class(topk):
    """
    Return the class with the highest count
    """
    return topk.group('Class').sort('count', descending=True).column(0).item(0)

def classify(training, example, k):
    """
    Return the majority class among the 
    k nearest neighbors of example
    """
    return majority_class(closest(training, example, k))

---

## Evaluation

---

### Example: Google Science Fair

<img src="./google_fair.png" width=60%>

[Brittany Wenger](https://edu.google.com/case-studies/brittany-wenger/), a 17-year-old high school student in 2012 won by building a breast cancer classifier with 99% accuracy. 


---

### Demo: Google Science Fair

* Load the `breast-cancer.csv` dataset, which contains cell measurements and a label: **benign (0)** or **malignant (1)**  
* The values were assigned manually by medical professionals after visually inspecting images
* Visualize the relationship between **Bland Chromatin** and **Single Epithelial Cell Size** features that will be used to try to classify the cells
    - The scatter plot may look like it has only a few points because many data points overlap
    - This happens because the measurements are not precise numbers but more like categories or rankings, leading to **many repeated values**
    - The `jittered` table adds random noise to the points to make them easier to see and avoid **overplotting**
    - [What is overplotting?](https://blogs.sas.com/content/iml/2011/07/05/jittering-to-prevent-overplotting-in-statistical-graphics.html#:~:text=Jittering%20is%20the%20act%20of,rounded%20to%20some%20convenient%20unit.)
* Split into three tables (train, validation, and test), where approximately 70% of the data is used for training, 15% is used for validation, and 15% is used for testing
* Create a function that evaluates the accuracy by returning the proportion of correctly classified examples in the test set
* Evaluate the accuracy for `k = 5`

In [ ]:
patients = (Table.read_table('breast-cancer.csv')
                 .drop('ID'))
patients.show(5)

In [ ]:
...

In [ ]:
patients.scatter('Bland Chromatin', 
                 'Single Epithelial Cell Size', 
                 group='Class')
plt.title('You are witnessing something called overplotting!')
plt.show()

In [ ]:
def randomize_column(a):
    return a + np.random.normal(0.0, 0.09, size=len(a))

jittered = Table().with_columns([
        'Bland Chromatin (jittered)', 
        randomize_column(patients.column('Bland Chromatin')),
        'Single Epithelial Cell Size (jittered)', 
        randomize_column(patients.column('Single Epithelial Cell Size')),
        'Class',
        patients.column('Class')
    ])

jittered.scatter(0, 1, group='Class')
plt.title('Jittering Helps Visually Overcome Overplotting')
plt.show()

In [ ]:
np.random.seed(1234) # Makes sure we all get the same data
row_70th_percentile = ...
train_bc, validation_and_test_bc = ...

In [ ]:
train_bc

In [ ]:
validation_and_test_bc

In [ ]:
np.random.seed(1234) # Makes sure we all get the same data
row_50th_percentile = ...
validation_bc, test_bc = ...

In [ ]:
validation_bc

In [ ]:
test_bc

In [ ]:
example_patient = test_bc.drop('Class').row(2)
example_patient

In [ ]:
train_bc.scatter('Bland Chromatin', 
                 'Single Epithelial Cell Size', 
                 group='Class')
plt.plot(example_patient.item('Bland Chromatin'), 
         example_patient.item('Single Epithelial Cell Size'), 
         marker='*', color='red', markersize=12)
plt.title('Training Data with Example Patient (Red Star)')
plt.show()

In [ ]:
...

In [ ]:
actual_label = test_bc.row(2).item('Class')
actual_label

In [ ]:
predicted_classes = ...
test_bc_attributes = ...
for ...:
    predicted_class = ...
    predicted_classes = ...

predicted_classes

In [ ]:
test_bc_with_predictions = ...
test_bc_with_predictions

In [ ]:
...

In [ ]:
def evaluate_accuracy(training, test, k):
    '''Return the proportion of correctly classified examples 
    in the test set'''
    test_attributes = ...
    predicted_classes = ...
    for i in np.arange(...):
        predicted_class = ...
        predicted_classes = ...

    test_with_predictions = ...
    accuracy = ...
    return ...

In [ ]:
...

---

## Picking `k`

- How can we decide on which value of `k` to use?
    - Evaluate a performance metric for various values of `k`
    - Determine the optimal `k` that is found by finding the smallest `k` with the best performance
- Do you use the test set for this?
    - No! The test is meant to represent unseen, future data and should only be used to evaluate the classifier
    - Instead, we use the validation set when deciding on a good value for `k`
        - Use the validation set to assess the "accuracy" for different values of `k`

---

### Demo: Picking `k`

- Calculate the accuracy on the validation set for several values of `k`.
- Visualize the trend in accuracy over the `k` to find the optimal value.

In [ ]:
train_bc.show(4)

In [ ]:
validation_bc.show(4)

In [ ]:
def which_k(training, validation, k_values):

    accuracies = make_array()

    for i in np.arange(np.size(k_values)):

        evaluate_accuracy(training, validation, k_values.item(i))
        accuracies = np.append(accuracies, 
                               evaluate_accuracy(training, validation, k_values.item(i)))

    k_table = Table().with_columns("k", k_values,
                                   "Accuracy", accuracies)
                             
    return k_table

In [ ]:
k_s = make_array(1,3,5,7,9,11,13,15,17,19,21,23,25) # Try many odd k values
k_table = which_k(train_bc, validation_bc, k_s)
k_table

In [ ]:
k_table.plot('k', 'Accuracy')
plt.title('The best k for the job is...?');

---

## Example: Chronic Kidney Disease (CKD)

---

### Demo: CKD Data Classification

* Load the `ckd` data using only Glucose and Hemoglobin features
* Split into three tables (train, validation, and test), where approximately 70% of the data is used for training, 15% is used for validation, and 15% is used for testing.
* Determine the optimal `k' using the validation set
* Evaluate the accuracy on the test set for the optimal `k`.
* Re-evaluate the accuracy if the White Blood Cell Count feature is included.

In [ ]:
ckd = (Table.read_table('ckd.csv')
       .relabeled('Blood Glucose Random', 'Glucose')
       .select('Glucose', 'Hemoglobin', 'Class'))
ckd

In [ ]:
np.random.seed(1234) # Makes sure we all get the same data
row_70th = round(ckd.num_rows * 0.70)
train_ckd, validation_and_test_ckd = ckd.split(row_70th)

In [ ]:
train_ckd

In [ ]:
validation_and_test_ckd

In [ ]:
np.random.seed(1234) # Makes sure we all get the same data
row_50th = ...
validation_ckd, test_ckd = ...

In [ ]:
validation_ckd

In [ ]:
test_ckd

In [ ]:
k_s = make_array(1,3,5,7,9,11,13,15,17,19,21,23,25) # Try many odd k values
k_table_ckd = which_k(train_ckd, validation_ckd, k_s)
k_table_ckd

In [ ]:
k_table_ckd.plot('k', 'Accuracy')
plt.title('The best k for the job is...?');

In [ ]:
...

In [ ]:
# Add White Blood Cell Count
ckd = (Table.read_table('ckd.csv')
       .relabeled('Blood Glucose Random', 'Glucose')
       .select('Glucose', 'Hemoglobin', 
               'White Blood Cell Count', 'Class'))
ckd

In [ ]:
# Same Train/Validation/Test Split with White Blood Cell Count added
np.random.seed(1234) # Makes sure we all get the same data
row_70th = round(ckd.num_rows * 0.70)
train_ckd, validation_and_test_ckd = ckd.split(row_70th)

In [ ]:
np.random.seed(1234) # Makes sure we all get the same data
row_50th = round(validation_and_test_ckd.num_rows*0.50)
validation_ckd, test_ckd = validation_and_test_ckd.split(row_50th)

In [ ]:
train_ckd.show(4)

In [ ]:
validation_ckd.show(4)

In [ ]:
test_ckd.show(4)

In [ ]:
...

---

## Standardizing Data

### When to Standardize Data

In [ ]:
ckd.show(4)

- If the attributes are on very different numerical scales, **distances can be distorted**
- As a result, classification can be **biased toward features with larger numerical values**
    - For example, in the `ckd` data, **Glucose** and **White Blood Cell Count** values are generally **orders of magnitude larger** than **Hemoglobin** values
    - This means our classification in the previous lecture was **more influenced by Glucose and White Blood Cell Count** than by Hemoglobin
    - In situations like this, it's a good idea to **convert all variables to standard units**
- **Standardizing** puts all numerical values on the **same scale**, making distance comparisons fair
- The best practice is to standardize the training, validation, and test set after splitting using the mean and standard deviation from the training set only
- If we standardize before we split, then we get data leakage because we would be using the data from the validation and test sets in the standardization process

---

### Demo: Standardizing Data

Explore what can happen to the classifier's accuracy (with 3 features) when the units are not standardized.

In [ ]:
accuracy_og_units = ...
accuracy_og_units

In [ ]:
def standardize_from_training(train, validation, test):
    """Standardize all columns (except 'Class') in train, validation, and test tables,
    using training set stats. Returns (train_su, validation_su, test_su)."""
    
    new_train_cols = []
    new_validation_cols = []
    new_test_cols = []
    
    for label in train.labels:
        if label == "Class":
            # Leave 'Class' column unchanged
            new_train_cols.append(train.column(label))
            new_validation_cols.append(validation.column(label))
            new_test_cols.append(test.column(label))
        else:
            train_values = train.column(label)
            mean_train = np.average(train_values)
            SD_train = np.std(train_values)
            
            # Standardize using training mean and SD
            new_train_cols.append((train_values - mean_train) / SD_train)
            new_validation_cols.append((validation.column(label)-mean_train)/ SD_train)
            new_test_cols.append((test.column(label) - mean_train) / SD_train)

    train_su = Table().with_columns(zip(train.labels, new_train_cols))
    validation_su = Table().with_columns(zip(validation.labels, new_validation_cols))
    test_su = Table().with_columns(zip(test.labels, new_test_cols))
    
    return train_su, validation_su, test_su

In [ ]:
train_su, validation_su, test_su = standardize_from_training(train_ckd, validation_ckd, test_ckd)

In [ ]:
train_su.show(4)

In [ ]:
validation_su.show(4)

In [ ]:
test_su.show(4)

In [ ]:
accuracy_su = ...
accuracy_su

In [ ]:
accuracies_ou = make_array()
accuracies_su = make_array()
num_iter = 50
for _ in np.arange(num_iter):
    train_ckd, validation_and_test_ckd = ckd.split(round(ckd.num_rows*0.7))
    validation_ckd, test_ckd = validation_and_test_ckd.split(round(validation_and_test_ckd.num_rows*0.5))
    train_su, validation_su, test_su = standardize_from_training(train_ckd, validation_ckd, test_ckd)
    accuracies_ou = np.append(accuracies_ou, evaluate_accuracy(train_ckd, test_ckd, 3))
    accuracies_su = np.append(accuracies_su, evaluate_accuracy(train_su, test_su, 3))

accuracies = Table().with_columns(
    'Iteration', np.arange(num_iter),
    'Original Units', accuracies_ou,
    'Standard Units', accuracies_su
)
accuracies.plot('Iteration')
plt.title('Accuracy for Random Train/Test Splits')
plt.show()

---

## Attribution

This content is licensed under the <a href="https://creativecommons.org/licenses/by-nc-sa/4.0/">Creative Commons Attribution-NonCommercial-ShareAlike 4.0 International License (CC BY-NC-SA 4.0)</a> and derived from the <a href="https://www.data8.org/">Data 8: The Foundations of Data Science</a> offered by the University of California, Berkeley.

<img src="./by-nc-sa.png" width=100px>